### Исследуем продвинутые модели - градиентный бустинг CatBoost

In [3]:
from catboost import CatBoostRegressor
import sys
from pathlib import Path
# чтобы убрать ошибку vs code
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src.preprocessing import CatBoostDataPreprocessor
import pandas as pd
from numpy import ndarray
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

In [4]:
df = pd.read_csv('../data/train.csv')
preprocessor = CatBoostDataPreprocessor()

data_train, data_test = train_test_split(df, test_size=0.2, random_state=42)

data_train = preprocessor.clean_train_data(data_train)

data_train_processed = preprocessor.fit_transform(data_train)
data_test_processed = preprocessor.transform(data_test)


y_train = data_train_processed[preprocessor.target_column]
X_train = data_train_processed.drop(columns=preprocessor.target_column)

y_test = data_test_processed[preprocessor.target_column]
X_test = data_test_processed.drop(columns=preprocessor.target_column)

### Подготовка функции для создания отчетов об метриках на выборках моделей:
- mae
- rmse
- mape
- r2

In [5]:
def count_metrics(y_true:ndarray|pd.Series, y_pred:ndarray)->dict:
    mae = mean_absolute_error(y_true,y_pred)
    rmse = root_mean_squared_error(y_true,y_pred)
    mape = mean_absolute_percentage_error(y_true,y_pred)
    r2 = r2_score(y_true,y_pred)
    return {
        "mae":mae,
        "rmse":rmse,
        "mape":mape,
        "r2":r2
    }

### Обучение модели

In [7]:
cb = CatBoostRegressor(
    iterations=412,
    learning_rate=0.05,
    # ОБЯЗАТЕЛЬНО УКАЗЫВАЕМ СТРОКОВЫЕ ФИЧИ!
    cat_features=preprocessor.cat_string_columns, 
    eval_metric='MAPE',
    random_seed=42,
    verbose=100
)

cb.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50
)

0:	learn: 0.3410719	test: 0.4002890	best: 0.4002890 (0)	total: 36.2ms	remaining: 14.9s
100:	learn: 0.0917725	test: 0.1239868	best: 0.1239868 (100)	total: 2.8s	remaining: 8.62s
200:	learn: 0.0771831	test: 0.1131843	best: 0.1131843 (200)	total: 5.37s	remaining: 5.64s
300:	learn: 0.0676378	test: 0.1094449	best: 0.1094449 (300)	total: 8.32s	remaining: 3.07s
400:	learn: 0.0602331	test: 0.1077373	best: 0.1077373 (400)	total: 11.9s	remaining: 327ms
411:	learn: 0.0594992	test: 0.1076507	best: 0.1076507 (411)	total: 12.2s	remaining: 0us

bestTest = 0.1076506601
bestIteration = 411



CatBoostRegressor(cat_features=['Foundation', 'GarageType', 'Electrical'], eval_metric='MAPE', iterations=412, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

### Анализируем метрики на train и на test:

In [8]:
predicts_train_regressor = cb.predict(X_train)
predicts_test_regressor = cb.predict(X_test)

metrics_train_regressor = count_metrics(y_train,predicts_train_regressor)
metrics_test_regressor = count_metrics(y_test,predicts_test_regressor)

print("Метрики CatBoost на train:\n ",metrics_train_regressor)
print("Метрики CatBoost на test:\n ",metrics_test_regressor)

Метрики CatBoost на train:
  {'mae': 9956.30041031999, 'rmse': 13615.670348908308, 'mape': 0.0602565715552288, 'r2': 0.9689697400704474}
Метрики CatBoost на test:
  {'mae': 17621.19999183037, 'rmse': 28321.446257787742, 'mape': 0.10765066012774405, 'r2': 0.8954276133156913}


### Выводы по исследованию моделей (Выбор финального решения):
В ходе данного этапа мы провели сравнение линейной модели (ElasticNet с подобранными гиперпараметрами) и алгоритма градиентного бустинга (CatBoost).
1. Сравнение качества предсказаний на тестовой (отложенной) выборке:
- MAE (Средняя абсолютная ошибка): Снизилась с 21530 до 17 621. Мы стали точнее предсказывать цену каждого дома в среднем почти на 4 000 долларов.
- MAPE (Средняя процентная ошибка): Упала с 12.4% до 10.7%.
- RMSE: Снизилась с 36 234 до 28 321 (модель стала гораздо реже допускать грубые ошибки на дорогих объектах).
- R2 (Коэффициент детерминации): Вырос с 0.83 до 0.895. Теперь наша модель способна объяснить почти 90% дисперсии в ценах на недвижимость.
2. Анализ переобучения (Train vs Test):
- Метрики CatBoost на тренировочной выборке феноменально высоки (R2 = 0.96, MAPE = 6%). Разрыв между Train и Test присутствует, что характерно для алгоритмов бустинга (склонность к переобучению). Однако, благодаря встроенному детектору переобучения (early_stopping_rounds=50), модель вовремя остановила рост деревьев (на 412-й итерации).
- Несмотря на разрыв между метриками Train/Test, тестовые метрики CatBoost абсолютно и безоговорочно превосходят метрики ElasticNet.
3. Бизнес-результат:
- CatBoost доказал свою высочайшую эффективность на табличных данных. Благодаря отказу от жесткого One-Hot Encoding и использованию CatBoostDataPreprocessor, мы позволили алгоритму самостоятельно находить сложные нелинейные зависимости в категориальных признаках.
4. Финальное решение для Production:
Для развертывания на бэкенде (FastAPI) утверждается связка:
- Класс препроцессинга: CatBoostDataPreprocessor
- Модель: CatBoostRegressor

In [9]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(preprocessor, '../models/catboost_preprocessor.pkl')
cb.save_model('../models/catboost_model.cbm')

print("Пайплайн CatBoost успешно сохранен и готов к деплою!")

Пайплайн CatBoost успешно сохранен и готов к деплою!
